In [0]:
import os
import pandas as pd
import numpy as np

In [0]:
facilities_path = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.facilities"
facilities = spark.read.table(facilities_path).toPandas()

postcodes_path = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.india_post_pincode_directory"
postcodes = spark.read.table(postcodes_path).toPandas()

health_indicators_path = "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset.nfhs_5_district_health_indicators"
health_indicators = spark.read.table(health_indicators_path).toPandas()

In [0]:
facilities.shape, postcodes.shape, health_indicators.shape

In [0]:
postcodes.head()

In [0]:
facilities.head()

In [0]:
health_indicators.head()

## Postcodes

In [0]:
postcodes.dtypes

In [0]:
import numpy as np

# Replace 'NA' strings with actual NaN values
facilities = facilities.replace('NA', np.nan).replace(1.0000000, np.nan)
postcodes = postcodes.replace('NA', np.nan).replace(1.0000000, np.nan)
health_indicators = health_indicators.replace('NA', np.nan)

print("Replaced 'NA' strings with NaN values in all dataframes")

In [0]:
postcodes.loc[pd.to_numeric(postcodes['latitude'], errors='coerce') < 2, ['latitude', 'longitude']] = [np.nan,np.nan]

In [0]:
postcodes.head()

In [0]:
postcodes.nunique()

In [0]:
postcodes.isna().sum()

In [0]:
display(postcodes['latitude'].value_counts().head())
display(postcodes['longitude'].value_counts().head())
postcodes['pincode'].value_counts().head()

In [0]:
postcodes['latitude'] = pd.to_numeric(postcodes['latitude'], errors='coerce')
postcodes['longitude'] = pd.to_numeric(postcodes['longitude'], errors='coerce')

In [0]:
postcodes_cleaned = postcodes.dropna().drop_duplicates(subset=['latitude', 'longitude'])

In [0]:
postcodes_cleaned['statename'].value_counts()

In [0]:
postcodes_cleaned['regionname'].value_counts()

## health_indicators

In [0]:
health_indicators_cleaned = health_indicators.copy()

In [0]:
health_indicators_cleaned = health_indicators_cleaned.replace('*', np.nan)

In [0]:
health_indicators_cleaned.head()

In [0]:
health_indicators_cleaned.loc[:, health_indicators_cleaned.select_dtypes(include='object').columns] = health_indicators_cleaned.loc[:, health_indicators_cleaned.select_dtypes(include='object').columns].apply(lambda col: col.str.replace('(', '', regex=False).str.replace(')', '', regex=False).str.replace(' ', '', regex=False))

In [0]:
health_indicators_cleaned[health_indicators_cleaned.select_dtypes(include='object').columns[2:]] = health_indicators_cleaned.loc[:, health_indicators_cleaned.select_dtypes(include='object').columns[2:]].astype(float)

In [0]:
health_indicators_cleaned.dtypes.value_counts()

In [0]:
health_indicators_cleaned.isna().sum().sort_values(ascending=False).head(20)

In [0]:
health_indicators_cleaned = health_indicators_cleaned.fillna(health_indicators_cleaned.median(numeric_only=True))

In [0]:
numeric_cols = health_indicators_cleaned.select_dtypes(include='number').columns
outlier_cols = []

for col in numeric_cols:
    q1 = health_indicators_cleaned[col].quantile(0.25)
    q3 = health_indicators_cleaned[col].quantile(0.75)
    iqr = q3 - q1
    outliers = health_indicators_cleaned[(health_indicators_cleaned[col] < (q1 - 1.5 * iqr)) | (health_indicators_cleaned[col] > (q3 + 1.5 * iqr))]
    if not outliers.empty:
        outlier_cols.append(col)

outlier_cols

In [0]:
# import os

# os.makedirs("data/processed", exist_ok=True)
# health_indicators_cleaned.to_csv("data/processed/health_indicators.csv")
# postcodes_cleaned.to_csv("data/processed/postcodes_cleaned.csv")

## Facilites

In [0]:
facilities_cleaned = facilities.copy()

In [0]:
facilities['specialties'].replace('null', None, inplace=True)

In [0]:
facilities_cleaned.dropna(inplace=True)

In [0]:
facilities_cleaned.isna().sum()

In [0]:
facilities['specialties'] = facilities['specialties'].apply(lambda x: eval(x) if isinstance(x, str) else x)

In [0]:
facilities['specialties'].replace('', None)

In [0]:
sp_un = facilities['specialties'].explode().replace('', None)

In [0]:
sp_un = sp_un.str.lower()
sp_un = sp_un.str.replace('and',' ').str.split(' ').explode()


In [0]:
# sp_un = sp_un.to_frame(name='specialty')
# sp_un['is_surgery'] = sp_un['specialty'].str.contains('surgery')

In [0]:
top_100_specialties = pd.Series(sp_un.value_counts().head(100).index)
top_100_specialties.to_csv('data/top_100_specialties.csv')

In [0]:
top_100_specialties

In [0]:
facilities_fin = facilities.copy()[['unique_id', 'numberDoctors', 'capacity', 'specialties', 'latitude', 'longitude']]

In [0]:
facilities_fin = facilities_fin.replace('null', np.nan)

In [0]:
for specialty in top_100_specialties:
    col_name = f"OHE_{specialty}"
    facilities_fin[col_name] = facilities_fin['specialties'].apply(
        lambda x: int(specialty in str(x).lower()) if x is not None else 0
    )
facilities_fin

In [0]:
facilities_fin.drop(columns=['specialties']).to_csv('data/processed/facilities.csv', index=False)